# Train Expert 802 on Kaggle's free GPU

This notebook trains the repository's **native Expert 802 Transformer from random initialization**. It does not call OpenAI, Gemini, Groq, or another generation API, and it does not silently substitute a pretrained model.

## Before running
1. In Kaggle Notebook settings, set **Accelerator = GPU** and enable **Internet** (needed to clone this public repository and install dependencies).
2. Add a Kaggle Dataset containing your reviewed, legally usable corpus. Its root must contain `corpus/` and `corpus_manifest.csv`. The CSV columns are `path,license,source_url,permission_basis,notes`; `path` is relative to `corpus/`.
3. Set `CORPUS_DATASET` below to the mounted dataset folder name. Do not upload private conversations, customer data, secrets, or material whose license does not permit model training.

Kaggle's free GPU availability and session duration are limited. Each run saves a resumable checkpoint bundle in `/kaggle/working/expert802-session.zip`. Download it from the notebook output, upload it as a **private Kaggle Dataset**, then attach that dataset to the next session as `expert802-previous-session`.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json, zipfile

# CHANGE THIS to the folder name shown under /kaggle/input for your corpus dataset.
CORPUS_DATASET = "expert802-licensed-corpus"
# Optional: attach your previous session ZIP as a Kaggle Dataset named this.
PREVIOUS_SESSION_DATASET = "expert802-previous-session"

# Training target is a TOTAL step count, not additional steps. Raise it on later sessions.
TOTAL_STEPS = 3000
BATCH_SIZE = 4
CONTEXT_LENGTH = 512
VOCAB_SIZE = 8000
START_FRESH = False  # Set True only when deliberately starting a new training run.

CORPUS_DIR = Path("/kaggle/input") / CORPUS_DATASET / "corpus"
MANIFEST = Path("/kaggle/input") / CORPUS_DATASET / "corpus_manifest.csv"
PREVIOUS_DIR = Path("/kaggle/input") / PREVIOUS_SESSION_DATASET
WORKSPACE = Path("/kaggle/working/expert_802_AI")
if not WORKSPACE.exists():
    subprocess.run(["git", "clone", "https://github.com/expertsteam802/expert_802_AI.git", str(WORKSPACE)], check=True)
os.chdir(WORKSPACE)
print("Workspace:", WORKSPACE)
print("Corpus exists:", CORPUS_DIR.exists(), "Manifest exists:", MANIFEST.exists())


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Kaggle Notebook settings choose Accelerator = GPU, then restart the session.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements.txt"], check=True)
subprocess.run([sys.executable, "training/test_model.py"], check=True)


In [ ]:
from pathlib import Path
import shutil, json, subprocess, sys, zipfile

if not CORPUS_DIR.is_dir() or not MANIFEST.is_file():
    raise FileNotFoundError(f"Expected {CORPUS_DIR} and {MANIFEST}. Check CORPUS_DATASET and your attached Kaggle Dataset layout.")

checkpoint_dir = WORKSPACE / "training" / "checkpoints"
data_dir = WORKSPACE / "training" / "data"
audit_file = WORKSPACE / "training" / "corpus_audit.json"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

if START_FRESH:
    shutil.rmtree(checkpoint_dir, ignore_errors=True)
    shutil.rmtree(data_dir, ignore_errors=True)

# Restore the previous private session bundle, if one was attached.
previous_zip = list(PREVIOUS_DIR.rglob("expert802-session.zip")) if PREVIOUS_DIR.exists() else []
if previous_zip and not START_FRESH:
    restore_dir = Path("/kaggle/working/expert802_restore")
    shutil.rmtree(restore_dir, ignore_errors=True)
    restore_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(previous_zip[0]) as archive:
        archive.extractall(restore_dir)
    restored = restore_dir / "checkpoints"
    if restored.is_dir():
        shutil.copytree(restored, checkpoint_dir, dirs_exist_ok=True)
    previous_audit = restore_dir / "corpus_audit.json"
else:
    previous_audit = None

# Audit the current corpus every session. Provenance is recorded; this is a screening gate, not legal advice.
subprocess.run([sys.executable, "training/audit_corpus.py", str(CORPUS_DIR), "--manifest", str(MANIFEST), "--output", str(audit_file)], check=True)
if previous_audit and previous_audit.exists():
    old = json.loads(previous_audit.read_text(encoding="utf-8"))
    new = json.loads(audit_file.read_text(encoding="utf-8"))
    old_signature = [(x["path"], x["sha256"], x["license"]) for x in old.get("accepted", [])]
    new_signature = [(x["path"], x["sha256"], x["license"]) for x in new.get("accepted", [])]
    if old_signature != new_signature:
        raise RuntimeError("The corpus differs from the previous session. Set START_FRESH=True to begin a separate run, or attach the exact same corpus.")

# Rebuild token files from the same corpus. Reuse the saved tokenizer on resumed runs so token IDs never drift.
saved_tokenizer = checkpoint_dir / "tokenizer.json"
prepare_cmd = [sys.executable, "training/prepare_data.py", str(CORPUS_DIR), "--output", str(data_dir), "--vocab-size", str(VOCAB_SIZE)]
if saved_tokenizer.exists():
    prepare_cmd += ["--tokenizer", str(saved_tokenizer)]
subprocess.run(prepare_cmd, check=True)
print("Prepared training data:", data_dir)


In [ ]:
train_cmd = [
    sys.executable, "training/train.py",
    "--data", str(data_dir), "--output", str(checkpoint_dir),
    "--steps", str(TOTAL_STEPS), "--batch-size", str(BATCH_SIZE),
    "--block-size", str(CONTEXT_LENGTH), "--layers", "6", "--heads", "8", "--width", "256",
    "--eval-interval", "100",
]
latest = checkpoint_dir / "latest.pt"
if latest.exists():
    train_cmd += ["--resume", str(latest)]
    print("Resuming from the saved checkpoint; target total steps:", TOTAL_STEPS)
else:
    print("Starting from random initialization; this is not a pretrained model.")
subprocess.run(train_cmd, check=True)

subprocess.run([
    sys.executable, "training/evaluate.py",
    "--checkpoint", str(checkpoint_dir / "best.pt"),
    "--data", str(data_dir),
    "--tokenizer", str(checkpoint_dir / "tokenizer.json"),
    "--output", str(checkpoint_dir / "evaluation.json"),
], check=True)


In [ ]:
# Export only model checkpoints, tokenizer, evaluation and corpus provenance report.
# The original corpus and tokenized training data are deliberately NOT bundled.
export_dir = Path("/kaggle/working/expert802_export")
shutil.rmtree(export_dir, ignore_errors=True)
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(checkpoint_dir, export_dir / "checkpoints")
shutil.copy2(audit_file, export_dir / "corpus_audit.json")
zip_path = Path("/kaggle/working/expert802-session.zip")
zip_path.unlink(missing_ok=True)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file in export_dir.rglob("*"):
        if file.is_file():
            archive.write(file, file.relative_to(export_dir))
print("Session bundle ready:", zip_path)
print("Download expert802-session.zip from the Kaggle notebook Output panel. Keep it private and attach it to your next session to resume.")


## Important expectations
- This is genuine training from random initialization, but the current 6-layer, 256-wide model is a small experimental model, not ChatGPT-level.
- A low training loss does not prove factual accuracy, reasoning ability, or working code. Review `checkpoints/evaluation.json` and test generated samples.
- Kaggle sessions can end or GPU access can be unavailable. Save the ZIP output after every session and attach it next time.
- Do not publish checkpoints or corpus files containing private or restricted material. Keep your Kaggle datasets private when required by their licenses.
